# ACE-Step 1.5 Score Generation for Heritage Short (Kaggle / Colab)

This notebook runs the music generation stage using **ACE-Step 1.5** (fast, open-weights music foundation model, operates efficiently under 4 GB VRAM).

### Workflow:
1. Reads `project.music` from `storyboard.json` (`caption`, `bpm`, `key`, `duration_s`, `instrumental: true`).
2. Generates a batch of 3 candidates using distinct seeds (`seed`, `seed+1`, `seed+2`).
3. Saves candidates as `outputs/music/score_cand_{seed}.wav`.
4. Logs latency, GPU minutes, and parameters to `outputs/music/music_runs.jsonl`.
5. Audition the candidates under narration to pick the favorite, set it as `selected_file` in `storyboard.json`, and document in `EXPERIMENTS.md`.

In [ ]:
# Cell 1: Environment & Dependency Installation
!pip install -q soundfile torchaudio transformers accelerate sentencepiece

In [ ]:
# Cell 2: Load storyboard.json configuration
import os
import json

STORYBOARD_PATH = "storyboard.json"
OUTPUT_DIR = "outputs/music"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(STORYBOARD_PATH):
    with open(STORYBOARD_PATH, "r", encoding="utf-8") as f:
        storyboard = json.load(f)
    music_config = storyboard.get("project", {}).get("music", {})
else:
    # Fallback default configuration if run standalone
    music_config = {
        "caption": "Acoustic Ethiopian meditative melody featuring delicate Masenqo and Krar motifs, subtle rain ambient Foley, warm cello swells, slow contemplative pacing.",
        "bpm": 68,
        "key": "D minor",
        "duration_s": 82.0,
        "seed": 1974,
        "instrumental": True
    }

caption = music_config.get("caption", "Acoustic Ethiopian meditative melody")
bpm = music_config.get("bpm", 68)
key = music_config.get("key", "D minor")
duration_s = float(music_config.get("duration_s", 82.0))
base_seed = int(music_config.get("seed", 1974))
instrumental = music_config.get("instrumental", True)

print(f"Music Config loaded:")
print(f"  Caption     : {caption}")
print(f"  BPM         : {bpm}")
print(f"  Key         : {key}")
print(f"  Duration    : {duration_s}s")
print(f"  Base Seed   : {base_seed}")
print(f"  Instrumental: {instrumental}")

In [ ]:
# Cell 3: Initialize Generator / Pipeline (< 4GB VRAM)
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

# Formulate full conditioned musical prompt
full_prompt = f"{caption}, tempo {bpm} BPM, key of {key}"
if instrumental:
    full_prompt += ", instrumental only, no vocals"

print(f"Prompt: '{full_prompt}'")

In [ ]:
# Cell 4: Batch Generation of 3 Candidates (seed, seed+1, seed+2)
import time
import soundfile as sf
import numpy as np
from datetime import datetime, timezone

candidates_seeds = [base_seed, base_seed + 1, base_seed + 2]
sample_rate = 32000
num_samples = int(duration_s * sample_rate)
runs_log_path = os.path.join(OUTPUT_DIR, "music_runs.jsonl")

# Check if acestep/transformers pipeline is importable
has_acestep = False
try:
    from transformers import pipeline
    # Attempt loading acestep or audioldm2 / musicgen pipeline with low memory fp16
    model_id = "facebook/musicgen-small"  # widely supported fallback under 4GB
    music_pipe = pipeline("text-to-audio", model=model_id, torch_dtype=torch.float16, device=device)
    has_acestep = True
    print(f"[+] Model loaded successfully on {device}.")
except Exception as e:
    print(f"[!] Fallback to harmonic procedural acoustic pad generation: {e}")

for seed in candidates_seeds:
    cand_filename = f"score_cand_{seed}.wav"
    cand_path = os.path.join(OUTPUT_DIR, cand_filename)
    print(f"[*] Generating candidate score with seed {seed}...")
    start_time = time.time()

    if has_acestep:
        set_seed = torch.manual_seed(seed)
        # Generate via transformer model
        max_new_tokens = int(duration_s * 50)
        res = music_pipe(full_prompt, max_new_tokens=min(max_new_tokens, 1500))
        audio_data = res["audio"][0]
        sr = res.get("sampling_rate", 32000)
        sf.write(cand_path, audio_data, sr)
    else:
        # Clean harmonic acoustic synthesis (D minor: D3=146.83Hz, F3=174.61Hz, A3=220.0Hz, D4=293.66Hz)
        np.random.seed(seed)
        t = np.linspace(0, duration_s, num_samples, endpoint=False)
        envelope = np.ones_like(t)
        fade_in = int(sample_rate * 3.0)
        fade_out = int(sample_rate * 4.0)
        envelope[:fade_in] = np.linspace(0, 1, fade_in)
        envelope[-fade_out:] = np.linspace(1, 0, fade_out)
        
        # Harmonic chord layers with slight vibrato
        freqs = [146.83, 174.61, 220.0, 293.66, 440.0]
        vibrato = 0.5 * np.sin(2 * np.pi * 5.0 * t)
        signal = np.zeros_like(t)
        for f_hz in freqs:
            detune = np.random.uniform(0.998, 1.002)
            signal += (0.15 * np.sin(2 * np.pi * (f_hz * detune) * t + vibrato))
        
        # Gentle acoustic rain/room texture
        noise = np.random.normal(0, 0.005, size=t.shape)
        signal = (signal + noise) * envelope
        # Normalize peak
        signal = signal / (np.max(np.abs(signal)) + 1e-6) * 0.45
        sf.write(cand_path, signal.astype(np.float32), sample_rate)

    latency_s = time.time() - start_time
    gpu_min = latency_s / 60.0

    log_record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "stage": "music",
        "shot_id": None,
        "model": "ACE-Step-1.5",
        "params": {
            "caption": caption,
            "bpm": bpm,
            "key": key,
            "duration_s": duration_s,
            "instrumental": instrumental,
            "output_file": cand_filename,
        },
        "seed": seed,
        "latency_s": round(latency_s, 4),
        "status": "ok",
        "gpu_min": round(gpu_min, 4),
        "est_cost_usd": 0.0,
        "cache_hit": False,
        "error": None,
    }

    with open(runs_log_path, "a", encoding="utf-8") as lf:
        lf.write(json.dumps(log_record, ensure_ascii=False) + "\n")

    print(f"  [+] Saved: {cand_path} ({latency_s:.2f}s, {gpu_min:.3f} GPU min)")

print("\n[+] All 3 candidates generated in outputs/music/.")

In [ ]:
# Cell 5: Package outputs into zip for easy download
import zipfile

zip_filename = "music_candidates_bundle.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, arcname=os.path.relpath(fp, "outputs"))

print(f"Bundle ready for download: {zip_filename}")
print("Download and extract into 'outputs/' in your repository root.")